# 01 - Audit dan pembersihan data (STEM-Err-ID)
Jalankan **sekali** dari atas ke bawah. Hasil: `sains_bersih.csv`, `tabel_deskripsi.csv`, `sampel_cek_kunci_manual.csv`, `laporan_audit.json` di Google Drive.
Setelah selesai, pakai notebook `02_evaluasi_model`. GPU tidak diperlukan di notebook ini.

In [ ]:
# 1. Konfigurasi
URL = "https://huggingface.co/datasets/indolem/IndoMMLU/resolve/main/IndoMMLU.csv"
OUT_DIR = "/content/drive/MyDrive/stem_err_id"

COL = dict(mapel="subject", jenjang="level", kelas="kelas",
           soal="soal", opsi="jawaban", kunci="kunci")

# Cocok per kata utuh, tidak peka huruf besar/kecil
KATA_SAINS = ["ipa", "fisika", "kimia", "biologi"]

In [ ]:
# 2. Baca data mentah
import os, re, ast, json
import pandas as pd
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(OUT_DIR, exist_ok=True)

df = pd.read_csv(URL)
print("Ukuran data mentah:", df.shape)
print("Kolom:", list(df.columns))
for k, v in COL.items():
    assert v in df.columns, f"Kolom '{v}' ({k}) tidak ada. Kolom yang ada: {list(df.columns)}"
print("\nSemua mapel di dataset:")
print(df[COL["mapel"]].value_counts().to_string())

Mounted at /content/drive
Ukuran data mentah: (14981, 11)
Kolom: ['id', 'sumber', 'keterangan sumber', 'kelas', 'nomor', 'subject', 'level', 'soal', 'jawaban', 'kunci', 'is_for_fewshot']

Semua mapel di dataset:
subject
Bahasa Indonesia           3213
Bahasa Sunda               1157
Bahasa Jawa                 992
IPA                         970
Biologi                     845
Agama Islam                 703
PPKN                        699
Kimia                       685
Kesenian                    601
IPS                         599
Sejarah                     498
Sosiologi                   496
Fisika                      495
Geografi                    490
Ekonomi                     488
Bahasa Bali                 471
Bahasa Madura               295
Agama Kristen               201
Budaya Alam Minangkabau     199
Bahasa Makassar             186
Agama Hindu                 150
Penjaskes                   148
Bahasa Lampung              147
Bahasa Banjar               144
Bahasa Dayak

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 3. Filter sains + parsing opsi
HURUF = ["A", "B", "C", "D", "E"]

def parse_opsi(x):
    x = str(x).strip()
    if x.startswith("[") and x.endswith("]"):
        try:
            return [str(o).strip() for o in ast.literal_eval(x)]
        except Exception:
            pass
    baris = [l.strip() for l in x.split("\n") if l.strip()]
    if len(baris) >= 2:
        return baris
    opsi_split = re.split(r"\s*(?:[A-Ea-e][\.\)])\s*", x)
    opsi_split = [o.strip() for o in opsi_split if o.strip()]
    if len(opsi_split) >= 2:
        return opsi_split
    return [x]

def bersih_opsi(o):
    return re.sub(r"^\s*[A-Ea-e][\.\)]\s*", "", o).strip()

pola = r"\b(?:" + "|".join(KATA_SAINS) + r")\b"
sains = df[df[COL["mapel"]].astype(str).str.contains(pola, case=False, regex=True)].copy()
print("Mapel yang tertangkap filter (PERIKSA: harus sains semua):")
print(sains[COL["mapel"]].value_counts().to_string())

sains["opsi_list"] = sains[COL["opsi"]].apply(lambda x: [bersih_opsi(o) for o in parse_opsi(x)])
sains["n_opsi"] = sains["opsi_list"].apply(len)
sains["kunci_n"] = sains[COL["kunci"]].astype(str).str.strip().str.upper().str[:1]

assert sains["kunci_n"].isin(HURUF).all(), f"Kunci bukan huruf A-E: {sains['kunci_n'].unique()}"
valid = sains.apply(lambda r: HURUF.index(r["kunci_n"]) < r["n_opsi"], axis=1)
n_kunci_di_luar_opsi = int((~valid).sum())
if n_kunci_di_luar_opsi:
    print(f"Peringatan: {n_kunci_di_luar_opsi} soal kunci di luar jumlah opsi, dibuang.")
    sains = sains[valid].copy()
n_awal = len(sains)
print("Jumlah opsi:", sains["n_opsi"].value_counts().to_dict())
print("Soal sains setelah parsing:", n_awal)

Mapel yang tertangkap filter (PERIKSA: harus sains semua):
subject
IPA        970
Biologi    845
Kimia      685
Fisika     495
Peringatan: 1 soal kunci di luar jumlah opsi, dibuang.
Jumlah opsi: {5: 1814, 4: 1176, 3: 4}
Soal sains setelah parsing: 2994


In [ ]:
# 4. Audit kualitas (format dan konsistensi, BUKAN kebenaran isi kunci)
s = sains.copy()
s["soal_norm"] = (s[COL["soal"]].astype(str).str.strip().str.lower()
                  .str.replace(r"\s+", " ", regex=True))
s["opsi_key"] = s["opsi_list"].apply(lambda o: "|".join(x.lower() for x in o))
g = s.groupby("soal_norm")["kunci_n"].nunique()

lap = {
    "n_sains_awal": n_awal,
    "kunci_di_luar_opsi_dibuang": n_kunci_di_luar_opsi,
    "soal_terlalu_pendek": int((s["soal_norm"].str.len() < 10).sum()),
    "ada_opsi_kosong": int(s["opsi_list"].apply(lambda o: any(len(x) == 0 for x in o)).sum()),
    "opsi_kembar_dalam_soal": int(s["opsi_list"].apply(lambda o: len({x.lower() for x in o}) < len(o)).sum()),
    "duplikat_soal_dan_opsi": int(s.duplicated(["soal_norm", "opsi_key"]).sum()),
    "soal_sama_kunci_beda": int((g > 1).sum()),
}
print("LAPORAN AUDIT:")
for k, v in lap.items():
    print(f"  {k}: {v}")

LAPORAN AUDIT:
  n_sains_awal: 2994
  kunci_di_luar_opsi_dibuang: 1
  soal_terlalu_pendek: 0
  ada_opsi_kosong: 0
  opsi_kembar_dalam_soal: 25
  duplikat_soal_dan_opsi: 51
  soal_sama_kunci_beda: 9


In [ ]:
# 5. Pembersihan
bersih = s[(s["soal_norm"].str.len() >= 10)
           & ~s["opsi_list"].apply(lambda o: any(len(x) == 0 for x in o))]
bersih = bersih.drop_duplicates(["soal_norm", "opsi_key"])
bersih = bersih[~bersih["soal_norm"].isin(g[g > 1].index)]   # kunci bertentangan -> buang
lap["n_sains_akhir"] = len(bersih)
print(f"Sebelum: {n_awal} soal | Sesudah: {len(bersih)} soal | Dibuang: {n_awal - len(bersih)}")

# Fitur sederhana (dipakai Anggota 2 untuk prediktor kerawanan)
teks = bersih[COL["soal"]].astype(str)
bersih["panjang_soal"] = teks.str.split().str.len()
bersih["n_angka"] = teks.str.count(r"\d+")
bersih["rujukan_visual"] = teks.str.contains(r"gambar|grafik|tabel|diagram", case=False).astype(int)

Sebelum: 2994 soal | Sesudah: 2925 soal | Dibuang: 69


In [ ]:
# 6. Simpan (id = indeks asli dataset)
out = bersih[[COL["mapel"], COL["jenjang"], COL["kelas"], COL["soal"], "opsi_list",
              "n_opsi", "kunci_n", "panjang_soal", "n_angka", "rujukan_visual"]].copy()
out.columns = ["mapel", "jenjang", "kelas", "soal", "opsi_list",
               "n_opsi", "kunci", "panjang_soal", "n_angka", "rujukan_visual"]
out["opsi_list"] = out["opsi_list"].apply(lambda o: json.dumps(o, ensure_ascii=False))
out.index.name = "id"
out.to_csv(f"{OUT_DIR}/sains_bersih.csv")
out.sample(30, random_state=1).to_csv(f"{OUT_DIR}/sampel_cek_kunci_manual.csv")
with open(f"{OUT_DIR}/laporan_audit.json", "w") as f:
    json.dump(lap, f, indent=2)

tabel = out.groupby(["mapel", "jenjang"]).agg(
    n_soal=("soal", "size"),
    rata_kata=("panjang_soal", "mean"),
    pct_ada_angka=("n_angka", lambda x: (x > 0).mean() * 100),
    pct_rujukan_visual=("rujukan_visual", lambda x: x.mean() * 100)).round(1)
tabel.to_csv(f"{OUT_DIR}/tabel_deskripsi.csv")
print("\nTABEL DESKRIPSI (untuk makalah):")
print(tabel.to_string())
print("\nTotal:", len(out))
print("Jumlah opsi:", out["n_opsi"].value_counts().to_dict())
kd = out["kunci"].value_counts(normalize=True).round(3).to_dict()
print("Distribusi kunci:", kd)
print("Akurasi tebakan acak (rata-rata 1/n_opsi):", round((1 / out["n_opsi"]).mean(), 3))
print("\nFile tersimpan di:", OUT_DIR)


TABEL DESKRIPSI (untuk makalah):
                     n_soal  rata_kata  pct_ada_angka  pct_rujukan_visual
mapel   jenjang                                                          
Biologi SMA             457       16.3           14.4                 0.4
        Seleksi PTN     388       17.3           27.8                 0.0
Fisika  SMA             283       23.5           59.0                 1.4
        SMP             164       20.2           49.4                 0.0
IPA     SD              487       11.5            5.1                 1.0
        SMP             463       22.1           30.9                 0.6
Kimia   SMA             287       18.6           50.9                 1.0
        Seleksi PTN     396       34.6           91.2                 0.0

Total: 2925
Jumlah opsi: {5: 1798, 4: 1123, 3: 4}
Distribusi kunci: {'B': 0.231, 'C': 0.229, 'A': 0.223, 'D': 0.217, 'E': 0.099}
Akurasi tebakan acak (rata-rata 1/n_opsi): 0.219

File tersimpan di: /content/drive/MyDrive/stem

In [ ]:
# 7. Rapikan Drive: pindahkan hasil pilot lama ke arsip
import glob, shutil
os.makedirs(f"{OUT_DIR}/arsip_pilot", exist_ok=True)
for f in glob.glob(f"{OUT_DIR}/hasil_*_pilot.csv"):
    shutil.move(f, f"{OUT_DIR}/arsip_pilot/" + os.path.basename(f))
    print("Dipindah:", os.path.basename(f))
print("Isi folder:", sorted(os.listdir(OUT_DIR)))

Dipindah: hasil_Qwen3.5-4B_0shot_pilot.csv
Isi folder: ['arsip_pilot', 'laporan_audit.json', 'sains_bersih.csv', 'sampel_cek_kunci_manual.csv', 'tabel_deskripsi.csv']
